In [1]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.ls_opt import logging as ls_opt_logging
from cardiac_electrophysiology.ls_opt import optimizer
from cardiac_electrophysiology.utils import analysis, visualization

In [8]:
posterior_settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/mean_angle_field_const.npy"),
        ground_truth_path=Path("../data/ground_truth_angle_field.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=1,
        tau=1,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

optimizer_settings = optimizer.LBFGSConfig(
    maximum_num_iterations=1000,
    relative_function_tolerance= 1e-6,
    relative_gradient_tolerance=1e-6,
    max_line_search_steps=100,
)
ls_opt_logger_settings = ls_opt_logging.LSOPTLoggerSettings(
    print_to_console=True,
    logfile_path= Path("../results/lsopt_logfile.log"),
)

In [9]:
posterior_builder = builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
visualization.visualize_data_points(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)

Widget(value='<iframe src="http://localhost:45461/index.html?ui=P_0x7fbd1f1e7250_6&reconnect=auto" class="pyvi…

In [10]:
initial_guess = additional_output.prior_mean_parameter
ls_optimizer = optimizer.LBFGSOptimizer(optimizer_settings, ls_opt_logger_settings)
map_result = ls_optimizer.run(
    initial_guess=initial_guess,
    loss_function=posterior.evaluate_cost,
    gradient_function=posterior.evaluate_gradient,
)
print(f"MAP estimation success: {map_result.success}")
print(f"Status message: {map_result.status_message}")
np.save("../results/map_estimate.npy", map_result.result)
np.save("../results/map_loss_history.npy", map_result.loss_history)
np.save("../results/map_gradient_norm_history.npy", map_result.gradient_norm_history)

| Iteration   | Time        | Loss        | Grad Norm   | 
---------------------------------------------------------
| +1.000e+00  | +3.341e+00  | +4.088e+07  | +1.438e+06  | 
| +2.000e+00  | +4.408e+00  | +1.621e+07  | +4.708e+05  | 
| +3.000e+00  | +5.476e+00  | +1.047e+07  | +3.156e+05  | 
| +4.000e+00  | +6.567e+00  | +4.755e+06  | +2.385e+05  | 
| +5.000e+00  | +7.566e+00  | +2.987e+06  | +1.910e+05  | 
| +6.000e+00  | +8.641e+00  | +2.023e+06  | +1.004e+05  | 
| +7.000e+00  | +9.770e+00  | +1.634e+06  | +2.045e+05  | 
| +8.000e+00  | +1.082e+01  | +1.347e+06  | +1.013e+05  | 
| +9.000e+00  | +1.186e+01  | +1.241e+06  | +7.445e+04  | 
| +1.000e+01  | +1.295e+01  | +1.131e+06  | +6.310e+04  | 
| +1.100e+01  | +1.408e+01  | +1.124e+06  | +1.460e+05  | 
| +1.200e+01  | +1.526e+01  | +9.132e+05  | +5.282e+04  | 
| +1.300e+01  | +1.634e+01  | +8.459e+05  | +4.687e+04  | 
| +1.400e+01  | +1.734e+01  | +7.030e+05  | +6.763e+04  | 
| +1.500e+01  | +1.835e+01  | +6.057e+05  | +4.477e+04  |

In [11]:
map_parameter = np.load("../results/map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

Prior mean angle L2-error: 98.74821205557917
Prior mean angle max-error: 1.5707538677573534
MAP angle L2-error: 93.70395755825714
MAP angle max-error: 1.5702962576151518
Prior mean predictive L2-error: 544.2200317382812
Prior mean predictive max-error: 8.373613357543945
MAP predictive L2-error: 36.996910095214844
MAP predictive max-error: 1.8654346466064453
Data predictive L2-error: 6.649985596156064
Data predictive max-error: 1.0869193840457392


In [12]:
for data in (
    analysis_data.prior_mean_parameter,
    analysis_data.ground_truth_parameter,
    analysis_data.map_parameter,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=True,
    )
for data in (
    analysis_data.diff_lat_truth_prior,
    analysis_data.diff_lat_truth_map,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=False,
    )

Widget(value='<iframe src="http://localhost:45461/index.html?ui=P_0x7fbc06d16ad0_7&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45461/index.html?ui=P_0x7fbd1f1e6850_8&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45461/index.html?ui=P_0x7fbbf4c21f90_9&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45461/index.html?ui=P_0x7fbbf4c23610_10&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:45461/index.html?ui=P_0x7fbbf4c23ed0_11&reconnect=auto" class="pyv…